# Ukrainian company revenue: data, panel and baselines

Forecast of next-year net revenue (line 2000) of Ukrainian companies from open annual financial statements.

**What this notebook does**
- Downloads annual financial statements for 2022–2024 (State Statistics Service, data.gov.ua). 2025 is the holdout and is never read.
- Parses ~1.1M XML reports into one row per company-year.
- Builds a modelling panel: target = next-year net revenue on a log scale, in 2021 prices.
- Sets up time-based validation folds and evaluates naive benchmarks N1–N3 and a panel AR(1) model (M1), followed by leakage checks.

Early file revisions are used because the current versions of the annual files hold only 1–4% of the reports.

**Outputs.** Hugging Face dataset [`Shnnapka/fin-zvit-ua`](https://huggingface.co/datasets/Shnnapka/fin-zvit-ua): `reports/` (one table per reporting year) and `panel/` (modelling panel and a 0.6% sample). To load the panel:

```python
panel = pl.read_parquet(hf_hub_download('Shnnapka/fin-zvit-ua', 'panel/panel_v1.parquet', repo_type='dataset'))
```

**How to run.** Restart & Run All with the default flags takes ~1–2 minutes and reads the published reports from Hugging Face.
`REBUILD = True` re-downloads and re-parses everything (~30 min, ~4.3 GB of archives). `UPLOAD = True` pushes the files to Hugging Face and asks for a write token (never stored).

**Sections:** 0. Data & target · 1. Data preparation · 2. Validation setup · 3. Baselines & metrics · 4. Leakage checks

# 0. Data & target

**Source:** data.gov.ua, State Statistics Service dataset “Фінансова звітність підприємств (річна)”, reporting years 2022–2024.

**Target:** z = log1p(net revenue at t+1), in 2021 prices. Population: companies with revenue y_t > 0 that filed a report for t+1.

## Setup

In [64]:
!pip -q install huggingface_hub
import io, os, re, subprocess, time, zipfile
from getpass import getpass

import numpy as np
import polars as pl
import requests
import statsmodels.api as sm
from huggingface_hub import HfApi, hf_hub_download, login

RAW, OUT = '/content/raw', '/content/out'
os.makedirs(RAW, exist_ok=True)
os.makedirs(OUT, exist_ok=True)

HF_REPO = 'Shnnapka/fin-zvit-ua'
REBUILD = False   # True: download and parse the raw XML again (~30 min)
UPLOAD = False    # True: upload reports and panel to HF_REPO (asks for a token)

BASE = 'https://data.gov.ua/dataset/6456ef0c-985c-4bb7-bf9d-16831161b416/resource'
API = 'https://data.gov.ua/api/3/action/resource_show'

# (file name, resource_id, revision_id or None = current version).
# The current versions of the annual files contain only late filings (1–4% of reports),
# so early revisions are pinned: 2022 = per-form files (Dec 2023 snapshot),
# 2023 = revision of Apr 2024, 2024 = revision of Apr 2025.
SOURCES = {
    2022: [
        ('rep2022_f1', '1e93dd50-f137-43f1-a165-6d7fd8f2c329', None),
        ('rep2022_f2', '4cee01d7-a483-4b28-88fb-4dfe770afe2c', None),
        ('rep2022_m',  '70d2d63c-a3c0-4818-bb7f-f6ab5854c7b0', None),
        ('rep2022_ms', '4d214aeb-b9ae-4d80-a220-3b585bf1e90c', None),
    ],
    2023: [('rep2023_apr2024', '9e7b6a7d-eaeb-44e2-9947-e34b25ce83cf', 397278)],
    2024: [('rep2024_apr2025', 'c58bf087-77b1-4355-9879-16d8cea35677', 442489)],
}

# Small/micro income statement: _3 = reporting year, _4 = previous year (see "Check column mapping").
SMALL_INCOME_CUR, SMALL_INCOME_PREV = '_3', '_4'

def load_parquet(name, hf_dir):
    """Read OUT/name; if missing, download HF_REPO/hf_dir/name and keep a local copy."""
    local = f'{OUT}/{name}'
    if not os.path.exists(local):
        path = hf_hub_download(HF_REPO, f'{hf_dir}/{name}', repo_type='dataset')
        pl.read_parquet(path).write_parquet(local)
    return pl.read_parquet(local)

_hf_api = None

def hf_upload(pattern, path_in_repo):
    """Upload files from OUT that match `pattern` to HF_REPO/path_in_repo. Does nothing unless UPLOAD is True."""
    global _hf_api
    if not UPLOAD:
        print(f'UPLOAD = False: skipped {path_in_repo}/{pattern}')
        return
    if _hf_api is None:
        login(token=getpass('Paste your HF token and press Enter: '))
        _hf_api = HfApi()
        _hf_api.create_repo(HF_REPO, repo_type='dataset', private=False, exist_ok=True)
    _hf_api.upload_folder(folder_path=OUT, repo_id=HF_REPO, repo_type='dataset',
                          path_in_repo=path_in_repo, allow_patterns=pattern)
    print('✓ Uploaded:', f'https://huggingface.co/datasets/{HF_REPO}/tree/main/{path_in_repo}')

## Download helper

In [65]:
def download(name, rid, rev):
    dst = f'{RAW}/{name}.zip'
    if os.path.exists(dst + '.ok'):
        return dst
    if rev is None:
        url = requests.get(API, params={'id': rid}, timeout=60).json()['result']['url']
    else:
        url = f'{BASE}/{rid}/revision/{rev}/download'
    print(f'   ↓ {name} ...', flush=True)
    subprocess.run(['wget', '-q', '-c', '-O', dst, url], check=True)
    zipfile.ZipFile(dst).close()            # fails if the archive is truncated
    open(dst + '.ok', 'w').close()
    print(f'   ✓ {name}: {os.path.getsize(dst) / 1e6:.0f} MB', flush=True)
    return dst

## Parse XML into one row per company-year

Line codes used (Ukrainian national accounting standards, НП(С)БО 1 and 25):

| Code | Field | Form |
|---|---|---|
| 1300 | `assets`: total assets | balance |
| 1495 | `equity`: total equity | balance |
| 1595 | `lt_liab`: long-term liabilities | balance |
| 1695 | `cur_liab`: current liabilities | balance |
| 2000 | `revenue`: net revenue (**target**) | income statement |
| 2350 / 2355 | `net_result`: net profit / net loss (full form: 2350 − 2355) | income statement |
| `S_CH` | `employees` (small/micro only) | header |

Column mapping (checked below): full balance `B` = end of year, `A` = start; full income statement `A` = current year, `B` = previous;
small/micro balance `_4` = end, `_3` = start; small/micro income statement `_3` = current, `_4` = previous.

In [66]:
FORM_CODES = [('S01001', 'f1'), ('S01002', 'f2'), ('S01100', 'm'), ('S01110', 'ms')]
TAG = re.compile(rb'<([A-Z0-9_]+)>([^<]*)</\1>')

def form_of(name):
    for code, form in FORM_CODES:
        if code in name:
            return form
    return None

def iter_xml(path):
    """Yield (form, XML bytes) from an archive, including nested zips."""
    with zipfile.ZipFile(path) as outer:
        for ii in outer.infolist():
            n = ii.filename
            if n.lower().endswith('.zip'):
                with zipfile.ZipFile(io.BytesIO(outer.read(ii))) as inner:
                    for jj in inner.infolist():
                        f = form_of(jj.filename)
                        if f and jj.filename.lower().endswith('.xml'):
                            yield f, inner.read(jj)
            else:
                f = form_of(n)
                if f and n.lower().endswith('.xml'):
                    yield f, outer.read(ii)

def parse(b):
    """Flat {tag: text} dict; empty if the file has no XML."""
    # 2022 files wrap the XML in a digital signature, so cut out the XML itself.
    s, e = b.find(b'<?xml'), b.rfind(b'</DECLAR>')
    if s < 0:
        return {}
    return {k.decode(): v.decode('utf-8', 'replace') for k, v in TAG.findall(b[s:e])}

In [67]:
def num(d, k):
    v = d.get(k)
    if v is None or v.strip() == '':
        return None
    try:
        return float(v.replace(',', '.'))
    except ValueError:
        return None

def diff(a, b):
    return None if a is None and b is None else (a or 0.0) - (b or 0.0)

def header(d):
    edr = (d.get('FIRM_EDRPOU') or d.get('TIN') or '').strip()
    fill = d.get('D_FILL', '')                      # DDMMYYYY -> YYYYMMDD
    fill = int(fill[4:8] + fill[2:4] + fill[0:2]) if len(fill) == 8 and fill.isdigit() else 0
    cnt = d.get('C_DOC_CNT', '')
    return {
        'edrpou': edr.zfill(8) if edr.isdigit() else edr,
        'year': int(d['PERIOD_YEAR']) if d.get('PERIOD_YEAR', '').isdigit() else None,
        'period_type': d.get('PERIOD_TYPE'),
        'd_fill': fill,
        'doc_cnt': int(cnt) if cnt.isdigit() else 0,
        'kved': (d.get('FIRM_KVED') or d.get('KVED') or '').strip() or None,
        'obl': (d.get('OBL') or d.get('C_REG') or '').strip() or None,
    }

def row(form, d):
    r = header(d)
    if form == 'f1':     # full balance: B = end of year, A = start
        r.update(assets=num(d, 'B1300'), assets_prev=num(d, 'A1300'),
                 equity=num(d, 'B1495'), equity_prev=num(d, 'A1495'),
                 lt_liab=num(d, 'B1595'), lt_liab_prev=num(d, 'A1595'),
                 cur_liab=num(d, 'B1695'), cur_liab_prev=num(d, 'A1695'),
                 n3=num(d, 'N3'))
    elif form == 'f2':   # full income statement: A = current year, B = previous
        r.update(revenue=num(d, 'A2000'), revenue_prev=num(d, 'B2000'),
                 net_result=diff(num(d, 'A2350'), num(d, 'A2355')),
                 net_result_prev=diff(num(d, 'B2350'), num(d, 'B2355')))
    else:                # small/micro: balance _4 = end, _3 = start
        c, p = SMALL_INCOME_CUR, SMALL_INCOME_PREV
        r.update(assets=num(d, 'A1300_4'), assets_prev=num(d, 'A1300_3'),
                 equity=num(d, 'A1495_4'), equity_prev=num(d, 'A1495_3'),
                 lt_liab=num(d, 'A1595_4'), lt_liab_prev=num(d, 'A1595_3'),
                 cur_liab=num(d, 'A1695_4'), cur_liab_prev=num(d, 'A1695_3'),
                 revenue=num(d, 'B2000' + c), revenue_prev=num(d, 'B2000' + p),
                 net_result=num(d, 'B2350' + c), net_result_prev=num(d, 'B2350' + p),
                 employees=num(d, 'S_CH'))
    return r

def latest(df):
    """One report per (EDRPOU, year): the latest filing."""
    return (df.sort(['edrpou', 'year', 'd_fill', 'doc_cnt'])
              .unique(subset=['edrpou', 'year'], keep='last', maintain_order=True))

## Build one table per reporting year

The full form comes as two files (balance f1, income statement f2), which are joined per company.
If a company filed more than one form for the same year, the latest filing wins.

In [68]:
def build_year(year):
    t0 = time.time()
    rows = {'f1': [], 'f2': [], 'm': [], 'ms': []}
    n = skipped = 0
    for name, rid, rev in SOURCES[year]:
        path = download(name, rid, rev)
        for form, b in iter_xml(path):
            d = parse(b)
            if not d:
                skipped += 1
                continue
            if d.get('PERIOD_TYPE') != '5':        # 5 = annual report
                continue
            rows[form].append(row(form, d))
            n += 1
            if n % 100_000 == 0:
                print(f'      {year}: read {n:,} reports ({time.time() - t0:.0f} s)', flush=True)

    dfs = {k: latest(pl.DataFrame(v, infer_schema_length=None)) for k, v in rows.items() if v}
    other = {k: df.filter(pl.col('year') != year).height for k, df in dfs.items()}
    dfs = {k: df.filter(pl.col('year') == year) for k, df in dfs.items()}
    for k in ['f1', 'f2', 'm', 'ms']:
        if k not in dfs:
            print(f'   ⚠ {year}: no reports of form {k}', flush=True)

    parts = []
    if 'f1' in dfs and 'f2' in dfs:
        full = (dfs['f1'].join(dfs['f2'].select(['edrpou', 'year', 'revenue', 'revenue_prev',
                                                 'net_result', 'net_result_prev', 'd_fill', 'kved']),
                               on=['edrpou', 'year'], how='full', coalesce=True, suffix='_f2')
                .with_columns(pl.max_horizontal('d_fill', 'd_fill_f2').alias('d_fill'),
                              pl.coalesce('kved', 'kved_f2').alias('kved'))
                .drop(['d_fill_f2', 'kved_f2']))
    else:
        full = dfs.get('f1', dfs.get('f2'))
    if full is not None:
        parts.append(full.with_columns(pl.lit('full').alias('form')))
    if 'm' in dfs:
        parts.append(dfs['m'].with_columns(pl.lit('small').alias('form')))
    if 'ms' in dfs:
        parts.append(dfs['ms'].with_columns(pl.lit('micro').alias('form')))

    allf = pl.concat(parts, how='diagonal_relaxed')
    sizes = {f: allf.filter(pl.col('form') == f).height for f in ['full', 'small', 'micro']}
    before = allf.height
    allf = latest(allf)
    allf.write_parquet(f'{OUT}/reports_{year}.parquet')
    print(f'✓ {year}: {allf.height:,} companies (full {sizes["full"]:,}, small {sizes["small"]:,}, '
          f'micro {sizes["micro"]:,}; cross-form duplicates {before - allf.height:,}; '
          f'other-year reports dropped {other}; files without XML {skipped:,}) — {time.time() - t0:.0f} s',
          flush=True)
    return allf

## Download and parse 2022–2024 reports

Only with `REBUILD = True` (~30 min). Otherwise the result of the last rebuild is read from Hugging Face.

In [69]:
if REBUILD:
    years = {y: build_year(y) for y in [2022, 2023, 2024]}
    pl.concat(list(years.values()), how='diagonal_relaxed').write_parquet(f'{OUT}/reports_all.parquet')
    for y in years:
        print(f'   reports_{y}.parquet: {os.path.getsize(f"{OUT}/reports_{y}.parquet") / 1e6:.1f} MB')
    rep = pl.read_parquet(f'{OUT}/reports_all.parquet')
else:
    print('REBUILD = False: skipped download and parsing, using reports_all.parquet')
    rep = load_parquet('reports_all.parquet', 'reports')
    years = {y: rep.filter(pl.col('year') == y) for y in [2022, 2023, 2024]}

print(rep.group_by('year', 'form').len().pivot(on='form', index='year', values='len')
         .with_columns(pl.sum_horizontal('full', 'small', 'micro').alias('total'))
         .select(['year', 'full', 'small', 'micro', 'total']).sort('year'))

REBUILD = False: skipped download and parsing, using reports_all.parquet
shape: (3, 5)
┌──────┬───────┬────────┬────────┬────────┐
│ year ┆ full  ┆ small  ┆ micro  ┆ total  │
│ ---  ┆ ---   ┆ ---    ┆ ---    ┆ ---    │
│ i64  ┆ u32   ┆ u32    ┆ u32    ┆ u32    │
╞══════╪═══════╪════════╪════════╪════════╡
│ 2022 ┆ 16840 ┆ 209042 ┆ 118103 ┆ 343985 │
│ 2023 ┆ 16208 ┆ 218733 ┆ 136453 ┆ 371394 │
│ 2024 ┆ 15969 ┆ 234940 ┆ 163991 ┆ 414900 │
└──────┴───────┴────────┴────────┴────────┘


**Takeaway.** The data cover 343,985 companies for 2022, 371,394 for 2023 and 414,900 for 2024; about 95% file the small or micro form.
Opendatabot counts 343,793 / 381,454 / 428,365. 2022 matches within 0.1%; 2023 and 2024 are 2.6% and 3.1% lower because the April snapshots miss late filings.

## Check column mapping

The “previous year” column of a year-Y report should equal the “current year” column of the same company's Y−1 report.

In [70]:
def check(cur, prev):
    j = cur.join(prev, on='edrpou', suffix='_y1')
    for form in ['full', 'small', 'micro']:
        s = j.filter(pl.col('form') == form)
        for a, b in [('revenue_prev', 'revenue_y1'), ('revenue', 'revenue_y1'),
                     ('assets_prev', 'assets_y1')]:
            ok = s.filter(pl.col(a).is_not_null() & pl.col(b).is_not_null())
            if ok.height:
                share = ok.filter((pl.col(a) - pl.col(b)).abs() < 0.6).height / ok.height  # values in thousand UAH, rounded
                print(f'   {form:6s} {a:13s}(Y) = {b:11s}(Y-1)?  {share:6.1%}  ({ok.height:,} companies)')

print('2024 vs 2023:'); check(years[2024], years[2023])
print('\n2023 vs 2022:'); check(years[2023], years[2022])

2024 vs 2023:
   full   revenue_prev (Y) = revenue_y1 (Y-1)?   96.4%  (12,567 companies)
   full   revenue      (Y) = revenue_y1 (Y-1)?    0.5%  (12,319 companies)
   full   assets_prev  (Y) = assets_y1  (Y-1)?   86.1%  (15,084 companies)
   small  revenue_prev (Y) = revenue_y1 (Y-1)?   97.4%  (139,686 companies)
   small  revenue      (Y) = revenue_y1 (Y-1)?    2.8%  (135,098 companies)
   small  assets_prev  (Y) = assets_y1  (Y-1)?   94.1%  (179,879 companies)
   micro  revenue_prev (Y) = revenue_y1 (Y-1)?   97.7%  (75,510 companies)
   micro  revenue      (Y) = revenue_y1 (Y-1)?    5.3%  (72,192 companies)
   micro  assets_prev  (Y) = assets_y1  (Y-1)?   95.0%  (118,152 companies)

2023 vs 2022:
   full   revenue_prev (Y) = revenue_y1 (Y-1)?   97.4%  (12,949 companies)
   full   revenue      (Y) = revenue_y1 (Y-1)?    0.4%  (12,662 companies)
   full   assets_prev  (Y) = assets_y1  (Y-1)?   90.3%  (15,469 companies)
   small  revenue_prev (Y) = revenue_y1 (Y-1)?   97.6%  (138,069 co

**Takeaway.** Previous-year revenue matches the prior report for 96–98% of companies in every form, while the current-year column matches only 0–5%.
So the column mapping is right. This also justifies taking 2021 values from the previous-year columns of the 2022 reports.

## Upload reports to Hugging Face

In [ ]:
hf_upload('reports_*.parquet', 'reports')

# 1. Data preparation

## Deflate to 2021 prices

Annual-average CPI (year over previous year), State Statistics Service. The 2024 value is still to be verified on ukrstat.gov.ua.

In [72]:
CPI_YOY = {2022: 1.202, 2023: 1.129, 2024: 1.065}
DEFL = {2021: 1.0}
for y in [2022, 2023, 2024]:
    DEFL[y] = DEFL[y - 1] * CPI_YOY[y]
print('Deflator (2021 prices = 1):', {k: round(v, 4) for k, v in DEFL.items()})

Deflator (2021 prices = 1): {2021: 1.0, 2022: 1.202, 2023: 1.3571, 2024: 1.4453}


## KVED division → section

In [73]:
# KVED-2010 division ranges for each section letter (same as NACE Rev. 2).
SECTIONS = [('A', 1, 3), ('B', 5, 9), ('C', 10, 33), ('D', 35, 35), ('E', 36, 39), ('F', 41, 43),
            ('G', 45, 47), ('H', 49, 53), ('I', 55, 56), ('J', 58, 63), ('K', 64, 66), ('L', 68, 68),
            ('M', 69, 75), ('N', 77, 82), ('O', 84, 84), ('P', 85, 85), ('Q', 86, 88), ('R', 90, 93),
            ('S', 94, 96), ('T', 97, 98), ('U', 99, 99)]

def kved_section(col='kved'):
    div = pl.col(col).str.extract(r'^(\d{1,2})', 1).cast(pl.Int32, strict=False)
    expr = pl.lit(None, dtype=pl.Utf8)
    for s, lo, hi in reversed(SECTIONS):
        expr = pl.when(div.is_between(lo, hi)).then(pl.lit(s)).otherwise(expr)
    return expr

## Company-year values, 2021 and revenue lag

- 2021 comes only from the previous-year columns of the 2022 reports (there is no 2021 file).
- For 2022–2024, previous-year columns of the t+1 report are never used: that file is not available at time t.
- The revenue lag (t−1) comes from the previous-year column of the year-t report, i.e. the same file as t, so it is allowed.

In [74]:
rep = load_parquet('reports_all.parquet', 'reports')
assert rep['year'].max() <= 2024, 'Data contains 2025 (holdout)!'

VARS = ['revenue', 'assets', 'equity', 'lt_liab', 'cur_liab', 'net_result']

own = (rep.select(['edrpou', 'year', 'form', 'kved', 'obl', 'employees'] + VARS)
          .with_columns(pl.lit('own').alias('src')))

prev21 = (rep.filter(pl.col('year') == 2022)
             .select(['edrpou', 'form', 'kved', 'obl'] + [pl.col(f'{v}_prev').alias(v) for v in VARS])
             .with_columns(pl.lit(2021).alias('year'), pl.lit(None, dtype=pl.Float64).alias('employees'),
                           pl.lit('prev_col_2022').alias('src')))

vals = pl.concat([own, prev21.select(own.columns)], how='vertical_relaxed')

deflator = pl.col('year').replace_strict(DEFL, return_dtype=pl.Float64)
vals = vals.with_columns([(pl.col(v) / deflator).alias(v) for v in VARS])

lag = (rep.select(['edrpou', 'year', 'revenue_prev'])
          .with_columns((pl.col('revenue_prev') /
                         (pl.col('year') - 1).replace_strict(DEFL, default=None, return_dtype=pl.Float64))
                        .alias('revenue_lag'))
          .select(['edrpou', 'year', 'revenue_lag']))
print(vals.group_by('year', 'src').len().sort('year'))

shape: (4, 3)
┌──────┬───────────────┬────────┐
│ year ┆ src           ┆ len    │
│ ---  ┆ ---           ┆ ---    │
│ i64  ┆ str           ┆ u32    │
╞══════╪═══════════════╪════════╡
│ 2021 ┆ prev_col_2022 ┆ 343985 │
│ 2022 ┆ own           ┆ 343985 │
│ 2023 ┆ own           ┆ 371394 │
│ 2024 ┆ own           ┆ 414900 │
└──────┴───────────────┴────────┘


## Build target and panel

One row per company and year t, with the target taken from the t+1 report.
A blank revenue line on a filed income statement counts as 0. A full-form report with a balance sheet but no income statement is dropped, because its revenue is unknown, not zero.
A negative lag becomes null.

In [75]:
INCOME = ['revenue', 'revenue_prev', 'net_result', 'net_result_prev']
no_income = (rep.filter((pl.col('form') == 'full') & pl.all_horizontal([pl.col(c).is_null() for c in INCOME]))
                .select(['edrpou', (pl.col('year') - 1).alias('year')]))

target = (vals.filter(pl.col('src') == 'own')
              .select(['edrpou', (pl.col('year') - 1).alias('year'),
                       pl.col('revenue').fill_null(0).alias('y_next'), pl.col('form').alias('form_next')])
              .filter(pl.col('year') >= 2021)
              .join(no_income, on=['edrpou', 'year'], how='anti'))

panel = (vals.filter(pl.col('year').is_between(2021, 2023))
             .join(lag, on=['edrpou', 'year'], how='left')
             .join(target, on=['edrpou', 'year'], how='inner')
             .filter(pl.col('revenue') > 0)
             .rename({'year': 't', 'revenue': 'y_t'})
             .with_columns(
                 (pl.col('t') + 1).alias('target_year'),
                 kved_section('kved').alias('kved_section'),
                 pl.col('y_t').log1p().alias('z_t'),
                 pl.when(pl.col('revenue_lag') >= 0).then(pl.col('revenue_lag').log1p()).alias('z_lag'),
                 pl.col('y_next').clip(lower_bound=0).log1p().alias('z_next'),
             )
             .with_columns((pl.col('z_next') - pl.col('z_t')).alias('dz_next'))
             .sort(['edrpou', 't']))

assert panel['target_year'].max() <= 2024, 'Target for 2025 is the holdout!'
assert (panel['t'] < panel['target_year']).all()

print(f'Full-form reports without an income statement (not used as targets): {no_income.height:,}')
print(panel.group_by('target_year').agg(pl.len().alias('rows'),
                                        pl.col('z_lag').is_null().mean().alias('share_no_lag'),
                                        pl.col('kved_section').is_null().mean().alias('share_no_kved'),
                                        pl.col('dz_next').median().alias('median_dz')).sort('target_year'))

Full-form reports without an income statement (not used as targets): 1,281
shape: (3, 5)
┌─────────────┬────────┬──────────────┬───────────────┬───────────┐
│ target_year ┆ rows   ┆ share_no_lag ┆ share_no_kved ┆ median_dz │
│ ---         ┆ ---    ┆ ---          ┆ ---           ┆ ---       │
│ i64         ┆ u32    ┆ f64          ┆ f64           ┆ f64       │
╞═════════════╪════════╪══════════════╪═══════════════╪═══════════╡
│ 2022        ┆ 233830 ┆ 1.0000       ┆ 0.0001        ┆ -0.4628   │
│ 2023        ┆ 219144 ┆ 0.0556       ┆ 0.0001        ┆ 0.0411    │
│ 2024        ┆ 225730 ┆ 0.0828       ┆ 0.0001        ┆ 0.0377    │
└─────────────┴────────┴──────────────┴───────────────┴───────────┘


**Takeaway.** Each target year has 219–234k companies. Target year 2022 has no lag because there is no 2020 data.
Median real revenue growth is −0.46 in logs for 2022 (≈ −37%, the war shock) and about +0.04 (≈ +4%) for 2023 and 2024. Dropping full-form reports without an income statement removes only 41 rows (21 for 2023, 20 for 2024).

## Save and upload panel

In [76]:
panel.write_parquet(f'{OUT}/panel_v1.parquet')

sample = panel.filter(pl.col('edrpou').hash(seed=42) % 1000 < 6)
sample.write_parquet(f'{OUT}/panel_v1_sample.parquet')

print(f"panel_v1: {panel.height:,} rows, {panel['edrpou'].n_unique():,} companies, "
      f"{os.path.getsize(f'{OUT}/panel_v1.parquet') / 1e6:.1f} MB")
print(f"sample:   {sample.height:,} rows, {sample['edrpou'].n_unique():,} companies")
print(panel.columns)

hf_upload('panel_v1*.parquet', 'panel')

panel_v1: 678,704 rows, 278,144 companies, 43.6 MB
sample:   4,083 rows, 1,677 companies
['edrpou', 't', 'form', 'kved', 'obl', 'employees', 'y_t', 'assets', 'equity', 'lt_liab', 'cur_liab', 'net_result', 'src', 'revenue_lag', 'y_next', 'form_next', 'target_year', 'kved_section', 'z_t', 'z_lag', 'z_next', 'dz_next']
✓ Uploaded: https://huggingface.co/datasets/Shnnapka/fin-zvit-ua/tree/main/panel


**Takeaway.** The panel has 678,704 rows for 278,144 companies. It is published on Hugging Face (`panel/panel_v1.parquet`), so every downstream notebook uses exactly the same rows.

# 2. Validation setup

Expanding window, horizon one year: **F1** trains on target 2022 and tests on 2023; **F2** trains on targets 2022–2023 and tests on 2024.

## Load panel and define folds

In [77]:
panel = load_parquet('panel_v1.parquet', 'panel')

FOLDS = {'F1': ([2022], 2023), 'F2': ([2022, 2023], 2024)}

def split(fold):
    train_years, test_year = FOLDS[fold]
    tr = panel.filter(pl.col('target_year').is_in(train_years))
    te = panel.filter(pl.col('target_year') == test_year)
    assert tr['target_year'].max() < test_year     # training targets strictly before the test year
    assert te['t'].max() < test_year                # test features come from years ≤ t
    return tr, te

## Companies per target year and form

In [78]:
print(panel.group_by('target_year', 'form').len().pivot(on='form', index='target_year', values='len')
           .sort('target_year'))

shape: (3, 4)
┌─────────────┬───────┬────────┬───────┐
│ target_year ┆ micro ┆ small  ┆ full  │
│ ---         ┆ ---   ┆ ---    ┆ ---   │
│ i64         ┆ u32   ┆ u32    ┆ u32   │
╞═════════════╪═══════╪════════╪═══════╡
│ 2022        ┆ 70155 ┆ 149902 ┆ 13773 │
│ 2023        ┆ 65652 ┆ 140390 ┆ 13102 │
│ 2024        ┆ 71241 ┆ 141849 ┆ 12640 │
└─────────────┴───────┴────────┴───────┘


**Takeaway.** Small-form companies are 63–64% of the panel in each year, micro 30–32% and full about 6%. The mix is stable, so the two folds compare like with like.

# 3. Baselines & metrics

## Metrics

`MAE_log` is the main metric (mean absolute error of z). `Bias` = mean(ẑ − z): negative means under-prediction.
`Skill` = 1 − MAE / MAE(N1), so a positive value beats persistence. `MdAPE` and `WAPE` are on the revenue scale; MdAPE uses only y > 0.

In [79]:
def metrics(z, z_hat, z_n1=None):
    z, z_hat = np.asarray(z, float), np.asarray(z_hat, float)
    y, y_hat = np.expm1(z), np.expm1(z_hat)
    mae = np.mean(np.abs(z - z_hat))
    pos = y > 0
    out = {
        'MAE_log': mae,
        'Bias': np.mean(z_hat - z),
        'MdAPE': np.median(np.abs(y[pos] - y_hat[pos]) / y[pos]),
        'WAPE': np.sum(np.abs(y - y_hat)) / np.sum(y),
        'n': len(z),
    }
    if z_n1 is not None:
        out['Skill'] = 1 - mae / np.mean(np.abs(z - np.asarray(z_n1, float)))
    return out

## Benchmarks N1–N3

- **N1** persistence: ẑ = z_t.
- **N2** sector drift: z_t plus the median growth z_t − z_lag of the company's KVED section in year t (country median if the section is missing).
- **N3** two-year mean: (z_t + z_lag) / 2, or z_t when there is no lag.

The N2 medians use year-t data only, which is known at time t.

In [80]:
def sector_growth(df):
    g = df.filter(pl.col('z_lag').is_not_null()).with_columns((pl.col('z_t') - pl.col('z_lag')).alias('dz_t'))
    by_sec = g.group_by('t', 'kved_section').agg(pl.col('dz_t').median().alias('g_sec'))
    by_all = g.group_by('t').agg(pl.col('dz_t').median().alias('g_all'))
    return by_sec, by_all

def predict_naive(te):
    by_sec, by_all = sector_growth(te)
    te = te.join(by_sec, on=['t', 'kved_section'], how='left').join(by_all, on='t', how='left')
    return te.with_columns(
        pl.col('z_t').alias('N1'),
        (pl.col('z_t') + pl.coalesce('g_sec', 'g_all', pl.lit(0.0))).alias('N2'),
        pl.when(pl.col('z_lag').is_not_null()).then((pl.col('z_t') + pl.col('z_lag')) / 2)
          .otherwise(pl.col('z_t')).alias('N3'),
    )

## Model M1: panel AR(1)

z_next = α + β·z_t, pooled over companies. Fitted with OLS (squared loss) and with LAD, i.e. median regression (absolute loss, which matches MAE_log). Both use all training rows.

In [81]:
def fit_m1(tr):
    X = sm.add_constant(tr['z_t'].to_numpy())
    return sm.OLS(tr['z_next'].to_numpy(), X).fit()

def fit_m1_lad(tr):
    X = sm.add_constant(tr['z_t'].to_numpy())
    return sm.QuantReg(tr['z_next'].to_numpy(), X).fit(q=0.5)

def predict_m1(model, te):
    return model.predict(sm.add_constant(te['z_t'].to_numpy(), has_constant='add'))

## Evaluate on F1 and F2

In [82]:
MODELS = ['N1', 'N2', 'N3', 'M1', 'M1-LAD']
rows, preds = [], {}
for fold in FOLDS:
    tr, te = split(fold)
    te = predict_naive(te)
    ols, lad = fit_m1(tr), fit_m1_lad(tr)
    te = te.with_columns(pl.Series('M1', predict_m1(ols, te)), pl.Series('M1-LAD', predict_m1(lad, te)))
    print(f'{fold}: train {tr.height:,}, test {te.height:,}')
    print(f'    M1      z_next = {ols.params[0]:.3f} + {ols.params[1]:.3f}·z_t   (R² = {ols.rsquared:.3f})')
    print(f'    M1-LAD  z_next = {lad.params[0]:.3f} + {lad.params[1]:.3f}·z_t')
    for name in MODELS:
        rows.append({'model': name, 'fold': fold, **metrics(te['z_next'], te[name], te['N1'])})
    preds[fold] = te

results = pl.DataFrame(rows)
summary = (results.group_by('model')
                  .agg(pl.col('MAE_log').mean().alias('MAE_log (mean F1–F2)'),
                       pl.col('Skill').min().alias('Skill (min)'))
                  .sort('MAE_log (mean F1–F2)'))
pl.Config.set_tbl_rows(30); pl.Config.set_float_precision(4)
print(results.select(['model', 'fold', 'MAE_log', 'Skill', 'Bias', 'MdAPE', 'WAPE', 'n']).sort('fold', 'model'))
print(summary)
results.write_parquet(f'{OUT}/results_A.parquet')

F1: train 233,830, test 219,144
    M1      z_next = -0.593 + 0.964·z_t   (R² = 0.591)
    M1-LAD  z_next = -0.335 + 0.984·z_t
F2: train 452,974, test 225,730
    M1      z_next = -0.330 + 0.966·z_t   (R² = 0.597)
    M1-LAD  z_next = -0.094 + 0.988·z_t
shape: (10, 8)
┌────────┬──────┬─────────┬─────────┬─────────┬────────┬────────┬────────┐
│ model  ┆ fold ┆ MAE_log ┆ Skill   ┆ Bias    ┆ MdAPE  ┆ WAPE   ┆ n      │
│ ---    ┆ ---  ┆ ---     ┆ ---     ┆ ---     ┆ ---    ┆ ---    ┆ ---    │
│ str    ┆ str  ┆ f64     ┆ f64     ┆ f64     ┆ f64    ┆ f64    ┆ i64    │
╞════════╪══════╪═════════╪═════════╪═════════╪════════╪════════╪════════╡
│ M1     ┆ F1   ┆ 1.4457  ┆ -0.4114 ┆ -0.5871 ┆ 0.6558 ┆ 0.7332 ┆ 219144 │
│ M1-LAD ┆ F1   ┆ 1.1875  ┆ -0.1593 ┆ -0.1747 ┆ 0.4990 ┆ 0.5552 ┆ 219144 │
│ N1     ┆ F1   ┆ 1.0243  ┆ 0.0000  ┆ 0.2745  ┆ 0.3468 ┆ 0.3744 ┆ 219144 │
│ N2     ┆ F1   ┆ 1.1518  ┆ -0.1245 ┆ -0.1191 ┆ 0.4717 ┆ 0.4815 ┆ 219144 │
│ N3     ┆ F1   ┆ 1.1324  ┆ -0.1055 ┆ 0.4638  ┆ 0.3873 ┆

**Takeaway.** No model beats persistence: N1 has the lowest MAE_log in both folds (1.024 for 2023, 0.902 for 2024).
M1 trained on 2022 learns the war shock (β = 0.964 < 1, negative intercept), so it under-predicts the recovery (Bias −0.59 in F1).
Switching OLS to LAD cuts M1's error by about 18% (1.446 → 1.188 and 1.171 → 0.950), but it is still worse than N1.

# 4. Leakage checks

## Panel and fold checks

Every check is an `assert`, so Run All stops if any of them fails.

In [83]:
def check_panel(panel):
    # 1. Features come only from years <= t, target is t+1
    assert (panel['t'] < panel['target_year']).all(), 'feature year >= target year'
    assert (panel['target_year'] - panel['t'] == 1).all(), 'horizon is not 1 year'
    # 2. Holdout 2025 is never used
    assert panel['target_year'].max() <= 2024 and panel['t'].max() <= 2023, '2025 found'
    # 3. One row per company-year
    assert panel.select(['edrpou', 't']).is_unique().all(), 'duplicate company-year rows'
    # 4. Previous-year columns of the t+1 file are never features:
    #    only t = 2021 comes from previous-year columns (of the 2022 reports)
    bad = panel.filter((pl.col('t') >= 2022) & (pl.col('src') != 'own')).height
    assert bad == 0, f'{bad} rows with t >= 2022 use previous-year columns'
    # 5. No "present in all years" filter: the panel must be unbalanced
    share_all = (panel.group_by('edrpou').len()['len'] == panel['t'].n_unique()).mean()
    assert share_all < 1, 'every firm appears in every year -> survivor filter?'
    print(f'Panel checks passed. Firms present in all {panel["t"].n_unique()} years: {share_all:.1%} '
          f'(the rest enter or exit, so there is no survivor filter).')

def check_folds(panel, folds, split):
    for fold, (train_years, test_year) in folds.items():
        tr, te = split(fold)
        assert tr['target_year'].max() < test_year, f'{fold}: train overlaps test in time'
        assert te['t'].max() < test_year, f'{fold}: test features from the test year'
        assert te.filter(pl.col('src') != 'own').height == 0, f'{fold}: test uses previous-year columns'
        print(f'{fold}: train targets {sorted(tr["target_year"].unique().to_list())}, '
              f'test target {test_year} — OK')

check_panel(panel)
check_folds(panel, FOLDS, split)

Panel checks passed. Firms present in all 3 years: 65.3% (the rest enter or exit, so there is no survivor filter).
F1: train targets [2022], test target 2023 — OK
F2: train targets [2022, 2023], test target 2024 — OK


**Takeaway.** All checks pass. Features always come from the year before the target, 2025 is never used, there is one row per company-year, and previous-year columns are used only for t = 2021 (and never in a test fold).
65.3% of companies appear in all three years; the other 34.7% enter or exit, so the panel is not limited to survivors.

## Preprocessing fitted on training data only

Helper for feature-engineering notebooks: after fitting a scaler or imputer, its statistics must equal the training-set statistics, not the train+test ones.

In [84]:
def check_fitted_on_train(name, fitted_values, X_train, X_all=None, atol=1e-6):
    """Call after fitting a scaler / imputer: its stats must equal the TRAIN stats.
    fitted_values: e.g. scaler.mean_ or imputer.statistics_
    X_all: optional train+test matrix — stats must differ from it."""
    X_train = np.asarray(X_train, float)
    stat = np.nanmedian if 'median' in name.lower() else np.nanmean
    assert np.allclose(fitted_values, stat(X_train, axis=0), atol=atol, equal_nan=True), \
        f'{name}: not fitted on train only'
    if X_all is not None:
        assert not np.allclose(fitted_values, stat(np.asarray(X_all, float), axis=0), atol=atol, equal_nan=True), \
            f'{name}: matches train+test'
    print(f'{name}: fitted on training rows only — OK')

# Usage:
#   scaler = StandardScaler().fit(X_tr)
#   check_fitted_on_train('scaler mean', scaler.mean_, X_tr, np.vstack([X_tr, X_te]))
#   imputer = SimpleImputer(strategy='median').fit(X_tr)
#   check_fitted_on_train('imputer median', imputer.statistics_, X_tr)

# Known limitation (not target leakage): the population and N2 sector medians use only firms
# that also report in t+1 (survivors). This follows from the target definition.